In [ ]:
import requests
import pandas as pd
import time
import re
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

START_DATE = "1980-01-01"
END_DATE = "2020-12-31"

# File di input con le coordinate e file di output meteo
INPUT_CSV = "anagrafica_comuni_final_coordinate.csv"
OUTPUT_CSV = "piemonte_montano_weather_1980_2020.csv"

# Pausa standard in secondi tra una richiesta e l'altra per l'API meteo
REQUEST_DELAY = 5

session = requests.Session()

retry_strategy = Retry(
    total=5,
    backoff_factor=10,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)
session.mount("http://", adapter)

###############################################################################
# FUNZIONE PER ESTRARRE LAT E LON DALLA STRINGA "POINT (LON LAT)"
###############################################################################
def estrai_coordinate(point_str):
    try:
        if pd.isna(point_str) or not isinstance(point_str, str):
            return None, None
        # Estrae i numeri decimali all'interno delle parentesi tonda
        numeri = re.findall(r"[-+]?\d*\.\d+|\d+", point_str)
        if len(numeri) >= 2:
            # Nel formato POINT l'ordine standard è (Longitudine, Latitudine)
            lon = float(numeri[0])
            lat = float(numeri[1])
            return lat, lon
    except Exception:
        pass
    return None, None

###############################################################################
# LETTURA E FILTRAGGIO AUTOMATICO COMUNI DAL NUOVO FILE
###############################################################################
print("Caricamento del dataset con coordinate in corso...")

# Carica il file usando il separatore ';'
df = pd.read_csv(INPUT_CSV, sep=';')

# Identifica l'ultima colonna che contiene la stringa 'POINT'
colonna_point = df.columns[-1]

print("Filtraggio dei comuni montani del Piemonte ed estrazione coordinate...")
# Applica il filtro Piemonte + Comune Montano
df_piemonte_montani = df[
    (df['Denominazione Regione'] == 'Piemonte') &
    (df['comune_montano'] == 1)
].copy()

# Applica la funzione di estrazione geometrica per generare le colonne numeriche 'lat' e 'lon'
df_piemonte_montani['lat'], df_piemonte_montani['lon'] = zip(*df_piemonte_montani[colonna_point].apply(estrai_coordinate))

# Rimuove righe che non hanno coordinate valide salvate
df_comuni_pronti = df_piemonte_montani.dropna(subset=['lat', 'lon'])

# Trasforma in lista per il ciclo di download: [Nome, Lat, Lon]
comuni_da_elaborare = df_comuni_pronti[['Denominazione in italiano', 'lat', 'lon']].values.tolist()

print(f"Trovati {len(comuni_da_elaborare)} comuni montani in Piemonte con coordinate pronte nel file.\n")

###############################################################################
# FUNZIONE DI DOWNLOAD DATI METEO
###############################################################################
def scarica_dati(nome, lat, lon):
    url = (
        "https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={lat}"
        f"&longitude={lon}"
        f"&start_date={START_DATE}"
        f"&end_date={END_DATE}"
        "&daily=temperature_2m_mean,rain_sum,snowfall_sum"
        "&timezone=Europe/Rome"
    )

    tentativi = 0
    max_tentativi = 5

    while tentativi < max_tentativi:
        try:
            response = session.get(url, timeout=120)

            if response.status_code == 429:
                wait_time = 30 * (tentativi + 1)
                print(f"429 per {nome} -> attendo {wait_time}s")
                time.sleep(wait_time)
                tentativi += 1
                continue

            response.raise_for_status()
            data = response.json()

            if "daily" not in data:
                print(f"Nessun dato meteo per {nome}")
                return None

            df_meteo = pd.DataFrame({
                "date": data["daily"]["time"],
                "temperature_mean": data["daily"]["temperature_2m_mean"],
                "rain_sum": data["daily"]["rain_sum"],
                "snowfall_sum": data["daily"]["snowfall_sum"]
            })

            df_meteo["date"] = pd.to_datetime(df_meteo["date"])
            df_meteo["year"] = df_meteo["date"].dt.year
            df_meteo["month"] = df_meteo["date"].dt.month

            monthly = (
                df_meteo.groupby(["year", "month"])
                .agg({
                    "temperature_mean": "mean",
                    "rain_sum": "sum",
                    "snowfall_sum": "sum"
                })
                .reset_index()
            )

            monthly["comune"] = nome
            return monthly

        except Exception as e:
            tentativi += 1
            print(f"Errore meteo {nome}: {e}")
            wait_time = 15 * tentativi
            print(f"Retry tra {wait_time}s")
            time.sleep(wait_time)

    print(f"Fallito definitivamente: {nome}")
    return None

###############################################################################
# LOOP PRINCIPALE DI DOWNLOAD
###############################################################################
all_data = []
conteggio = 0

for nome, lat, lon in comuni_da_elaborare:
    conteggio += 1
    print(f"[{conteggio}/{len(comuni_da_elaborare)}] Elaborazione: {nome} (Lat: {lat}, Lon: {lon})")

    # Avvia direttamente il download usando le coordinate lette dal file locale
    df_meteo_comune = scarica_dati(nome, lat, lon)

    if df_meteo_comune is not None:
        all_data.append(df_meteo_comune)

        # Salvataggio progressivo di sicurezza
        temp_df = pd.concat(all_data, ignore_index=True)
        temp_df.to_csv(OUTPUT_CSV, index=False)
        print(f"-> Salvataggio parziale aggiornato nel file CSV.")

    time.sleep(REQUEST_DELAY)

###############################################################################
# EXPORT FINALE ORDINATO
###############################################################################
if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    final_df = final_df[
        [
            "comune",
            "year",
            "month",
            "temperature_mean",
            "rain_sum",
            "snowfall_sum"
        ]
    ]
    final_df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nOperazione conclusa! File finale salvato: {OUTPUT_CSV}")
else:
    print("Nessun dato meteo scaricato. Controlla che la macchina abbia accesso a Internet per la chiamata ad Open-Meteo.")